# **Fine-tuning Dataset Creation**

## **Dataset Trimming**


In [ ]:
# ================================================================
# UNIFIED DATASET TRIMMING — 50K balanced samples
# Multi-turn integrity preserved + order maintained
# ================================================================

import json
import random
from collections import defaultdict
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

random.seed(42)

UNIFIED_PATH  = "/content/unified_dataset/unified_train.json"
TRIMMED_PATH  = "/content/unified_dataset/unified_train_61k.json"
TRIMMED_JSONL = "/content/unified_dataset/unified_train_61k.jsonl"

# ── Targets ───────────────────────────────────────────────────
TARGET = {
    "ANSWER" : 20_000,   # 33%
    "ASK"    : 23_000,   # 37%
    "ABSTAIN": 18_000,   # 30%
}
TARGET_TOTAL = 61_000

# ── Multi-turn cap rules ──────────────────────────────────────
def get_turn_cap(dialogue_length):
    if dialogue_length <= 7:  return dialogue_length   # keep all
    if dialogue_length <= 15: return 10
    if dialogue_length <= 30: return 15
    if dialogue_length <= 50: return 25
    return 35                                           # 51+


# ================================================================
# STEP 1 — LOAD + SEPARATE MULTI vs SINGLE TURN
# ================================================================

with open(UNIFIED_PATH) as f:
    all_samples = json.load(f)

print(f"Total samples loaded: {len(all_samples)}")

# Separate by turn type
multi_turn_samples  = [s for s in all_samples
                       if s["metadata"].get("multi_turn", False)
                       and s["metadata"].get("dialogue_id")]
single_turn_samples = [s for s in all_samples
                       if not s["metadata"].get("multi_turn", False)
                       or not s["metadata"].get("dialogue_id")]

print(f"Multi-turn samples  : {len(multi_turn_samples)}")
print(f"Single-turn samples : {len(single_turn_samples)}")


# ================================================================
# STEP 2 — GROUP MULTI-TURN BY DIALOGUE + APPLY CAPS
# ================================================================

# Group by dialogue_id — preserve order via turn_id
dialogues = defaultdict(list)
for s in multi_turn_samples:
    dialogues[s["metadata"]["dialogue_id"]].append(s)

# Sort each dialogue by turn_id
for dlg_id in dialogues:
    dialogues[dlg_id].sort(
        key=lambda x: x["metadata"].get("turn_id") or 0
    )
print(f"\nUnique dialogues     : {len(dialogues)}")

# Apply cap — keep only first N turns per dialogue
capped_dialogues  = {}
cap_stats         = defaultdict(int)   # original_len → count

for dlg_id, turns in dialogues.items():
    orig_len  = len(turns)
    cap       = get_turn_cap(orig_len)
    capped    = turns[:cap]
    capped_dialogues[dlg_id] = capped

    # Track cap category for stats
    if orig_len <= 7:    cap_stats["1-7 (kept all)"]    += 1
    elif orig_len <= 15: cap_stats["8-15 (cap 10)"]     += 1
    elif orig_len <= 30: cap_stats["16-30 (cap 15)"]    += 1
    elif orig_len <= 50: cap_stats["31-50 (cap 25)"]    += 1
    else:                cap_stats["51+ (cap 35)"]       += 1

capped_multi = [s for turns in capped_dialogues.values()
                for s in turns]

print(f"\nAfter capping:")
print(f"  Multi-turn samples retained : {len(capped_multi)}")
print(f"  Multi-turn samples removed  : "
      f"{len(multi_turn_samples) - len(capped_multi)}")
print(f"\n  Cap category breakdown:")
for cat, cnt in sorted(cap_stats.items()):
    print(f"    {cat:25}: {cnt} dialogues")


# ================================================================
# CORRECTED STEP 3+4 — Dialogue-level sampling
# NEVER split a dialogue — always take whole or none
# ================================================================

# All retained after capping
all_retained = capped_multi + single_turn_samples

# ── For multi-turn: work at DIALOGUE level not sample level ──
# Each dialogue has a "dominant action" = most frequent action in it
# We pick dialogues to hit our targets, then take ALL their turns

dialogue_meta = {}
for dlg_id, turns in capped_dialogues.items():
    action_counts_dlg = defaultdict(int)
    for t in turns:
        action_counts_dlg[t["action"]] += 1
    dominant = max(action_counts_dlg, key=action_counts_dlg.get)
    source   = turns[0]["metadata"]["source"]
    dialogue_meta[dlg_id] = {
        "dominant_action": dominant,
        "action_counts"  : dict(action_counts_dlg),
        "num_turns"      : len(turns),
        "source"         : source,
        "turns"          : turns
    }

# Group dialogues by dominant action
dlg_by_action = defaultdict(list)
for dlg_id, meta in dialogue_meta.items():
    dlg_by_action[meta["dominant_action"]].append(dlg_id)

# Shuffle
for act in dlg_by_action:
    random.shuffle(dlg_by_action[act])

print("Dialogues by dominant action:")
for act, dlgs in dlg_by_action.items():
    total_turns = sum(dialogue_meta[d]["num_turns"] for d in dlgs)
    print(f"  {act:10}: {len(dlgs):5} dialogues | "
          f"{total_turns:6} total turns")

# ── Single turn by action (HotpotQA, ContractNLI, ShARC standalone)
single_by_action = defaultdict(list)
for s in single_turn_samples:
    single_by_action[s["action"]].append(s)

# ── Now sample whole dialogues until we hit targets ───────────


# Source minimum guarantees
SOURCE_MINIMUMS = {
    "contract_nli": 7_000,   # use all available
    "hotpotqa"    : 14_000,
    "sharc"       : 14_000,
    "quac"        : 26_000,
}

selected_turns  = []
action_budget   = dict(TARGET)
source_counts_s = defaultdict(int)

# First pass: guarantee source minimums
print("\nFirst pass — guaranteeing source minimums...")

# ContractNLI (single-turn, mostly ABSTAIN/ANSWER)
cnli_pool = [s for s in single_turn_samples
             if s["metadata"]["source"] == "contract_nli"]
random.shuffle(cnli_pool)
cnli_take = cnli_pool[:SOURCE_MINIMUMS["contract_nli"]]
selected_turns.extend(cnli_take)
for s in cnli_take:
    action_budget[s["action"]] = max(
        0, action_budget[s["action"]] - 1)
    source_counts_s["contract_nli"] += 1
print(f"  contract_nli: {len(cnli_take)} samples added")

# HotpotQA (single-turn ANSWER)
hotpot_pool = [s for s in single_turn_samples
               if s["metadata"]["source"] == "hotpotqa"]
random.shuffle(hotpot_pool)
hotpot_take = hotpot_pool[:SOURCE_MINIMUMS["hotpotqa"]]
selected_turns.extend(hotpot_take)
for s in hotpot_take:
    action_budget[s["action"]] = max(
        0, action_budget[s["action"]] - 1)
    source_counts_s["hotpotqa"] += 1
print(f"  hotpotqa    : {len(hotpot_take)} samples added")

already_selected_dlg_ids = set(
    s["metadata"].get("dialogue_id","")
    for s in selected_turns
    if s["metadata"].get("dialogue_id")
)

# ShARC dialogues (dominant ASK signal)
# ShARC — pull from BOTH single_turn AND multi_turn pools
sharc_single = [s for s in single_turn_samples
                if s["metadata"]["source"] == "sharc"]
sharc_multi_dlgs = [d for d in (dlg_by_action.get("ASK",[]) +
                                dlg_by_action.get("ANSWER",[]) +
                                dlg_by_action.get("ABSTAIN",[]))
                    if dialogue_meta[d]["source"] == "sharc"]

random.shuffle(sharc_single)
random.shuffle(sharc_multi_dlgs)

sharc_turns_added = 0

# First take single-turn ShARC
for s in sharc_single:
    if sharc_turns_added >= SOURCE_MINIMUMS["sharc"]:
        break
    selected_turns.append(s)
    sharc_turns_added += 1
    source_counts_s["sharc"] += 1
    action_budget[s["action"]] = max(
        0, action_budget[s["action"]] - 1)

# Then top up with multi-turn ShARC dialogues if needed
for dlg_id in sharc_multi_dlgs:
    if sharc_turns_added >= SOURCE_MINIMUMS["sharc"]:
        break
    turns = dialogue_meta[dlg_id]["turns"]
    selected_turns.extend(turns)
    sharc_turns_added += len(turns)
    source_counts_s["sharc"] += len(turns)
    already_selected_dlg_ids.add(dlg_id)
    for t in turns:
        action_budget[t["action"]] = max(
            0, action_budget[t["action"]] - 1)

print(f"  sharc       : {sharc_turns_added} samples added")

# Second pass: fill remaining budget with QuAC dialogues
print("\nSecond pass — filling remaining budget with QuAC...")
# Update already_selected after ShARC
already_selected_dlg_ids = set(
    s["metadata"].get("dialogue_id","")
    for s in selected_turns
    if s["metadata"].get("dialogue_id")
)


# All remaining QuAC dialogues not yet selected
remaining_quac = [
    d for d in (dlg_by_action.get("ANSWER",[]) +
                dlg_by_action.get("ASK",[]) +
                dlg_by_action.get("ABSTAIN",[]))
    if dialogue_meta[d]["source"] == "quac"
    and d not in already_selected_dlg_ids
]
random.shuffle(remaining_quac)

total_budget = sum(action_budget.values())
print(f"  Remaining budget: {total_budget} turns across actions")
print(f"  Available QuAC dialogues: {len(remaining_quac)}")

quac_added = 0
QUAC_HARD_CAP = SOURCE_MINIMUMS["quac"]   # add this line

for dlg_id in remaining_quac:
    if sum(action_budget.values()) <= 0:
        break
    if quac_added >= QUAC_HARD_CAP:        # add this check
        break
    turns    = dialogue_meta[dlg_id]["turns"]
    dominant = dialogue_meta[dlg_id]["dominant_action"]
    if action_budget.get(dominant, 0) <= 0:
        continue
    selected_turns.extend(turns)
    quac_added += len(turns)
    source_counts_s["quac"] += len(turns)
    for t in turns:
        action_budget[t["action"]] = max(
            0, action_budget[t["action"]] - 1)

print(f"  QuAC turns added: {quac_added}")

# ── Fast trim — precompute dialogue sizes ─────────────────────
final_samples = selected_turns

if len(final_samples) > TARGET_TOTAL:
    # Precompute dialogue → [sample_ids] map once
    dlg_to_ids = defaultdict(list)
    single_ids = []
    for s in final_samples:
        d = s["metadata"].get("dialogue_id")
        if d:
            dlg_to_ids[d].append(s["id"])
        else:
            single_ids.append(s["id"])

    # Walk dialogues in order, keep whole dialogues until limit
    keep_ids = set()

    # Add single-turn first (they're small, deterministic)
    for sid in single_ids:
        if len(keep_ids) < TARGET_TOTAL:
            keep_ids.add(sid)

    # Add whole dialogues
    seen_dlg_order = []
    seen_dlg_set   = set()
    for s in final_samples:
        d = s["metadata"].get("dialogue_id")
        if d and d not in seen_dlg_set:
            seen_dlg_order.append(d)
            seen_dlg_set.add(d)

    for dlg_id in seen_dlg_order:
        ids = dlg_to_ids[dlg_id]
        if len(keep_ids) + len(ids) <= TARGET_TOTAL:
            keep_ids.update(ids)
        else:
            break   # stop adding whole dialogues

    final_samples = [s for s in final_samples
                     if s["id"] in keep_ids]

print(f"After trim: {len(final_samples)} samples")

# Re-sort by source → dialogue → turn
final_samples.sort(key=lambda x: (
    x["metadata"]["source"],
    x["metadata"].get("dialogue_id") or x["id"],
    x["metadata"].get("turn_id") or 0
))

# ── Final stats ───────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  FINAL DATASET SUMMARY")
print(f"{'='*55}")
print(f"  Total samples: {len(final_samples)}")

action_final = defaultdict(int)
source_final = defaultdict(int)
mt_final     = 0

for s in final_samples:
    action_final[s["action"]] += 1
    source_final[s["metadata"]["source"]] += 1
    if s["metadata"].get("multi_turn"):
        mt_final += 1

print(f"\n  Action distribution:")
for act in ["ANSWER","ASK","ABSTAIN"]:
    n   = action_final[act]
    pct = 100*n/len(final_samples)
    print(f"  {act:10}: {n:6}  ({pct:.1f}%)")

print(f"\n  Source distribution:")
for src, cnt in sorted(source_final.items()):
    pct = 100*cnt/len(final_samples)
    print(f"  {src:15}: {cnt:6}  ({pct:.1f}%)")

print(f"\n  Multi-turn : {mt_final} ({100*mt_final/len(final_samples):.1f}%)")

# ── Integrity check ───────────────────────────────────────────
dlg_check = defaultdict(list)
for s in final_samples:
    if s["metadata"].get("dialogue_id"):
        dlg_check[s["metadata"]["dialogue_id"]].append(
            s["metadata"].get("turn_id") or 0
        )

broken = 0
for dlg_id, turn_ids in dlg_check.items():
    sorted_ids = sorted([t for t in turn_ids if t is not None])
    if not sorted_ids:
        continue
    for i in range(len(sorted_ids)-1):
        if sorted_ids[i+1] - sorted_ids[i] > 1:
            broken += 1
            break

print(f"\n  Dialogue integrity:")
print(f"  Total dialogues : {len(dlg_check)}")
print(f"  Broken sequences: {broken} "
      f"({'✅ clean' if broken==0 else '⚠️ check'})")

# Save
with open(TRIMMED_PATH, "w") as f:
    json.dump(final_samples, f, indent=2)
with open(TRIMMED_JSONL, "w") as f:
    for s in final_samples:
        f.write(json.dumps(s) + "\n")

print(f"\nSaved → {TRIMMED_PATH}")
print(f"Saved → {TRIMMED_JSONL}")


# ================================================================
# STEP 7 — INTERACTIVE DIFF VIEWER
# Shows original vs trimmed side by side
# ================================================================

# Build lookup for original
orig_by_id     = {s["id"]: s for s in all_samples}
trimmed_ids    = set(s["id"] for s in final_samples)
removed_ids    = set(s["id"] for s in all_samples) - trimmed_ids

# Sample removed dialogues to show what was cut
removed_samples     = [s for s in all_samples
                       if s["id"] in removed_ids]
removed_by_action   = defaultdict(list)
for s in removed_samples:
    removed_by_action[s["action"]].append(s)

# Build dialogue-level diff
orig_dialogues_len  = defaultdict(int)
trim_dialogues_len  = defaultdict(int)
for s in all_samples:
    if s["metadata"].get("dialogue_id"):
        orig_dialogues_len[s["metadata"]["dialogue_id"]] += 1
for s in final_samples:
    if s["metadata"].get("dialogue_id"):
        trim_dialogues_len[s["metadata"]["dialogue_id"]] += 1

# Find capped dialogues (those that changed)
capped_examples = {
    dlg_id: (orig_dialogues_len[dlg_id], trim_dialogues_len[dlg_id])
    for dlg_id in trim_dialogues_len
    if trim_dialogues_len[dlg_id] < orig_dialogues_len[dlg_id]
}

print(f"\n  Dialogues that were capped: {len(capped_examples)}")


# ── Widgets ───────────────────────────────────────────────────
view_mode = widgets.ToggleButtons(
    options     = ["Trimmed Samples", "Removed Samples",
                   "Capped Dialogues"],
    description = "View:",
    style       = {"description_width":"50px",
                   "button_width":"160px"}
)
action_dd = widgets.Dropdown(
    options     = ["all","ANSWER","ASK","ABSTAIN"],
    value       = "ASK",
    description = "Action:",
    style       = {"description_width":"70px"},
    layout      = widgets.Layout(width="220px")
)
source_dd = widgets.Dropdown(
    options     = ["all","quac","sharc","hotpotqa","contract_nli"],
    value       = "all",
    description = "Source:",
    style       = {"description_width":"70px"},
    layout      = widgets.Layout(width="220px")
)
idx_slider = widgets.IntSlider(
    value=0, min=0, max=50, step=1,
    description="Item:",
    style       = {"description_width":"60px"},
    layout      = widgets.Layout(width="400px")
)
out = widgets.Output()

ACT_COLOR = {"ANSWER":"#00e676","ASK":"#40c4ff","ABSTAIN":"#ff5252"}

def render_sample_card(s, badge=""):
    act   = s["action"]
    color = ACT_COLOR.get(act,"#fff")
    ctx   = s["context"]["documents"][0]["text"][:250] \
            .replace("\n"," ") \
            if s["context"]["documents"] else ""
    return f"""
    <div style="font-family:monospace; font-size:12px;
                background:#1e1e1e; color:#e0e0e0; padding:12px;
                border-radius:8px; border:1px solid #444;
                margin-bottom:8px;">
      <div style="margin-bottom:5px;">
        <b style="color:#90caf9;">{s['id']}</b>
        &nbsp;|&nbsp; {s['metadata']['source']}
        &nbsp;|&nbsp; turn {s['metadata'].get('turn_id','—')}
        &nbsp;|&nbsp;
        <span style="color:{color}; font-weight:bold;">{act}</span>
        {f'&nbsp;<span style="background:#444; padding:1px 6px; border-radius:4px; font-size:10px;">{badge}</span>' if badge else ''}
      </div>
      <div style="background:#263238; padding:6px; border-radius:4px;
                  color:#cfd8dc; font-size:11px; margin-bottom:6px;">
        {ctx}...
      </div>
      <div style="color:#fff176; margin-bottom:4px;">
        <b>Q:</b> {s['query']}
      </div>
      <div style="color:#c8e6c9; font-size:11px;">
        <b>A:</b> {s['response'][:150]}
      </div>
    </div>"""


def render_capped_dialogue(dlg_id, orig_len, trim_len):
    turns_orig = sorted(
        [s for s in all_samples
         if s["metadata"].get("dialogue_id") == dlg_id],
        key=lambda x: x["metadata"].get("turn_id",0)
    )
    turns_kept = [s for s in final_samples
                  if s["metadata"].get("dialogue_id") == dlg_id]

    src = turns_orig[0]["metadata"]["source"] if turns_orig else "—"

    html = f"""
    <div style="font-family:monospace; font-size:12px;
                background:#1e1e1e; color:#e0e0e0; padding:12px;
                border-radius:8px; border:1px solid #555;">
      <div style="margin-bottom:8px;">
        <b style="color:#90caf9;">Dialogue:</b> {dlg_id[:50]}...
        &nbsp;|&nbsp; <b style="color:#90caf9;">Source:</b> {src}
        &nbsp;|&nbsp;
        <span style="color:#ff5252;">
          {orig_len} turns originally
        </span>
        &nbsp;→&nbsp;
        <span style="color:#00e676;">
          {trim_len} turns kept
        </span>
        &nbsp;
        <span style="color:#ffcc80;">
          ({orig_len-trim_len} removed)
        </span>
      </div>"""

    for t in turns_orig[:orig_len]:
        t_id   = t["metadata"].get("turn_id", 0)
        kept   = t_id <= trim_len
        act    = t["action"]
        color  = ACT_COLOR.get(act,"#fff")
        bg     = "#1b3a2b" if kept else "#3a1b1b"
        badge  = "✅ KEPT" if kept else "❌ REMOVED"
        bc     = "#00e676" if kept else "#ff5252"
        html  += f"""
      <div style="background:{bg}; padding:6px; margin:3px 0;
                  border-radius:4px; border-left:3px solid {bc};">
        <span style="color:{bc}; font-size:10px;">{badge}</span>
        &nbsp;
        <span style="color:#aaa;">Turn {t_id}</span>
        &nbsp;|&nbsp;
        <span style="color:{color};">{act}</span>
        &nbsp;|&nbsp;
        <span style="color:#fff176;">{t['query']}</span>
      </div>"""

    html += "</div>"
    return html


def update(change=None):
    mode   = view_mode.value
    action = action_dd.value
    source = source_dd.value

    with out:
        clear_output(wait=True)

        if mode == "Trimmed Samples":
            pool = [s for s in final_samples
                    if (action == "all" or s["action"] == action)
                    and (source == "all" or
                         s["metadata"]["source"] == source)]
            idx_slider.max = max(0, len(pool)-1)
            if not pool:
                print("No samples match.")
                return
            s   = pool[min(idx_slider.value, len(pool)-1)]
            print(f"Showing {min(idx_slider.value+1,len(pool))} "
                  f"of {len(pool)} trimmed samples")
            display(HTML(render_sample_card(s, badge="IN DATASET")))

        elif mode == "Removed Samples":
            pool = [s for s in removed_samples
                    if (action == "all" or s["action"] == action)
                    and (source == "all" or
                         s["metadata"]["source"] == source)]
            idx_slider.max = max(0, len(pool)-1)
            if not pool:
                print("No removed samples match.")
                return
            s   = pool[min(idx_slider.value, len(pool)-1)]
            print(f"Showing {min(idx_slider.value+1,len(pool))} "
                  f"of {len(pool)} removed samples")
            display(HTML(render_sample_card(
                s, badge="REMOVED")))

        elif mode == "Capped Dialogues":
            cap_list = sorted(capped_examples.items(),
                              key=lambda x: x[1][0]-x[1][1],
                              reverse=True)
            # Filter by source
            if source != "all":
                cap_list = [
                    (d, v) for d,v in cap_list
                    if any(s["metadata"]["source"]==source
                           for s in final_samples
                           if s["metadata"].get("dialogue_id")==d)
                ]
            idx_slider.max = max(0, len(cap_list)-1)
            if not cap_list:
                print("No capped dialogues match.")
                return
            dlg_id, (orig_l, trim_l) = \
                cap_list[min(idx_slider.value, len(cap_list)-1)]
            print(f"Capped dialogue "
                  f"{min(idx_slider.value+1,len(cap_list))} "
                  f"of {len(cap_list)}")
            display(HTML(render_capped_dialogue(
                dlg_id, orig_l, trim_l)))


view_mode.observe(update, names="value")
action_dd.observe(update, names="value")
source_dd.observe(update, names="value")
idx_slider.observe(update, names="value")

display(widgets.VBox([
    widgets.HTML(
        "<b style='color:#90caf9; font-size:13px;'>"
        "✂️ Dataset Trimming Viewer</b>"
    ),
    view_mode,
    widgets.HBox([action_dd, source_dd, idx_slider]),
    out
]))

update()

In [ ]:
# ================================================================
# VARIABLE POPULATION — known_variables + missing_variables
# Using GPT-4o-mini via OpenAI API
# Incremental saving — crash-safe
# ================================================================

!pip install openai tqdm ipywidgets -q

import json, os, time
from concurrent.futures import ThreadPoolExecutor, as_completed
import openai
from tqdm import tqdm
from collections import defaultdict
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── Config ────────────────────────────────────────────────────
OPENAI_API_KEY    = "sk-...."
openai.api_key    = OPENAI_API_KEY
client            = openai.OpenAI(api_key=OPENAI_API_KEY)

UNIFIED_PATH      = "/content/unified_dataset/unified_train_61k.json"
POPULATED_PATH    = "/content/unified_dataset/unified_train_populated.json"
CHECKPOINT_PATH   = "/content/unified_dataset/population_checkpoint.json"

N_TO_PROCESS      = 60995
BATCH_SIZE        = 50     # increased from 20
SLEEP_BETWEEN     = 0.1    # reduced from 0.3


# ================================================================
# LOAD DATA
# ================================================================

with open(UNIFIED_PATH) as f:
    all_samples = json.load(f)

print(f"Total samples: {len(all_samples)}")
print(f"Will process : {N_TO_PROCESS}")

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        checkpoint = json.load(f)
    print(f"✅ Checkpoint found — {len(checkpoint)} samples already done")
else:
    checkpoint = {}
    print("No checkpoint — starting fresh")


# ================================================================
# PROMPT TEMPLATES PER ACTION
# ================================================================

SYSTEM_PROMPT = """You are a variable extraction assistant for a QA system.

Given a query, context, and the action taken (ANSWER/ASK/ABSTAIN), extract:

1. known_variables: entities, concepts, or attributes EXPLICITLY present in the query
   - These are concrete nouns, named entities, or specific attributes mentioned
   - Keep them short (1-3 words each)
   - Max 5 items

2. missing_variables: entities, concepts, or attributes REQUIRED to resolve the query
   but NOT present in the query
   - For ANSWER: always empty []
   - For ASK: what clarification is needed
   - For ABSTAIN: what information is absent from context

Return ONLY valid JSON in this format:
{
  "known_variables": ["var1", "var2"],
  "missing_variables": ["var1", "var2"]
}

No explanation. No markdown. Only the JSON object."""


def build_user_prompt(sample):
    ctx_text = ""
    docs = sample["context"]["documents"]
    if docs:
        ctx_text = docs[0]["text"][:400]

    return f"""Query: {sample['query']}
Context (first 400 chars): {ctx_text}
Action: {sample['action']}
Response: {sample['response'][:200]}

Extract known_variables and missing_variables."""


# ================================================================
# EXTRACTION FUNCTIONS
# ================================================================

def extract_variables_single(sample):
    try:
        resp = client.chat.completions.create(
            model       = "gpt-4o-mini",
            messages    = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": build_user_prompt(sample)}
            ],
            max_tokens  = 150,
            temperature = 0.0,
        )
        raw = resp.choices[0].message.content.strip()
        raw = raw.replace("```json","").replace("```","").strip()
        parsed = json.loads(raw)

        known   = parsed.get("known_variables", [])
        missing = parsed.get("missing_variables", [])

        if sample["action"] == "ANSWER":
            missing = []

        known   = [str(v).strip() for v in known   if v][:5]
        missing = [str(v).strip() for v in missing if v][:8]

        return known, missing, raw

    except Exception as e:
        return [], [], f"ERROR: {e}"


def extract_variables_batch(batch_samples):
    combined = ""
    for i, s in enumerate(batch_samples):
        ctx = s["context"]["documents"][0]["text"][:300] \
              if s["context"]["documents"] else ""
        combined += f"""
--- SAMPLE {i+1} ---
Query: {s['query']}
Context: {ctx}
Action: {s['action']}
Response: {s['response'][:150]}
"""

    batch_system = SYSTEM_PROMPT + f"""

You will receive {len(batch_samples)} samples numbered 1 to {len(batch_samples)}.
Return a JSON array with one object per sample in order:
[
  {{"known_variables": [...], "missing_variables": [...]}},
  ...
]
"""
    try:
        resp = client.chat.completions.create(
            model      = "gpt-4o-mini",
            messages   = [
                {"role": "system", "content": batch_system},
                {"role": "user",   "content": combined}
            ],
            max_tokens  = 150 * len(batch_samples),
            temperature = 0.0,
        )
        raw = resp.choices[0].message.content.strip()
        raw = raw.replace("```json","").replace("```","").strip()
        parsed = json.loads(raw)

        results = []
        for i, (s, p) in enumerate(zip(batch_samples, parsed)):
            known   = [str(v).strip() for v in
                       p.get("known_variables",[]) if v][:5]
            missing = [str(v).strip() for v in
                       p.get("missing_variables",[]) if v][:8]
            if s["action"] == "ANSWER":
                missing = []
            results.append((known, missing))
        return results

    except Exception as e:
        results = []
        for s in batch_samples:
            k, m, _ = extract_variables_single(s)
            results.append((k, m))
        return results


# ================================================================
# MAIN POPULATION LOOP
# ================================================================

def select_samples(all_samples, n, checkpoint):
    not_done = [s for s in all_samples if s["id"] not in checkpoint]

    by_action = defaultdict(list)
    for s in not_done:
        by_action[s["action"]].append(s)

    selected = []
    per_action = n // 3

    for action in ["ASK", "ABSTAIN", "ANSWER"]:
        pool = by_action[action]
        selected.extend(pool[:per_action])

    remaining = n - len(selected)
    if remaining > 0:
        all_remaining = [s for s in not_done if s not in selected]
        selected.extend(all_remaining[:remaining])

    return selected[:n]


samples_to_process = select_samples(all_samples, N_TO_PROCESS, checkpoint)
print(f"\nSelected {len(samples_to_process)} samples to process")
act_dist = defaultdict(int)
for s in samples_to_process:
    act_dist[s["action"]] += 1
print(f"  ANSWER : {act_dist['ANSWER']}")
print(f"  ASK    : {act_dist['ASK']}")
print(f"  ABSTAIN: {act_dist['ABSTAIN']}")


# ── Process in parallel batches ───────────────────────────────
print(f"\nProcessing with parallel workers...")

batches = [
    samples_to_process[i:i+BATCH_SIZE]
    for i in range(0, len(samples_to_process), BATCH_SIZE)
]

def process_batch(batch_samples):
    to_do = [s for s in batch_samples if s["id"] not in checkpoint]
    if not to_do:
        return []
    results = extract_variables_batch(to_do)
    return list(zip(to_do, results))

new_done = 0
errors   = 0
MAX_WORKERS = 5   # drop to 3 if you hit rate limit errors

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_batch, b): b for b in batches}
    for i, future in enumerate(tqdm(as_completed(futures),
                                     total=len(batches),
                                     desc="Populating")):
        pairs = future.result()
        for sample, (known, missing) in pairs:
            checkpoint[sample["id"]] = {
                "known_variables"  : known,
                "missing_variables": missing
            }
            new_done += 1

        if i % 10 == 0:
            with open(CHECKPOINT_PATH, "w") as f:
                json.dump(checkpoint, f)

        time.sleep(0.05)

# Final checkpoint save
with open(CHECKPOINT_PATH, "w") as f:
    json.dump(checkpoint, f)

print(f"\n✅ Done. Processed {new_done} new samples.")
print(f"   Total in checkpoint: {len(checkpoint)}")
print(f"   Errors: {errors}")


# ================================================================
# MERGE CHECKPOINT INTO FULL DATASET
# ================================================================

print("\nMerging checkpoint into full dataset...")

populated_samples = []
updated = 0
skipped = 0

for sample in all_samples:
    s = dict(sample)

    if s["id"] in checkpoint:
        vars_data = checkpoint[s["id"]]
        s["state"]["known_variables"]   = vars_data["known_variables"]
        s["state"]["missing_variables"] = vars_data["missing_variables"]
        s["metadata"]["num_missing_variables"] = \
            len(vars_data["missing_variables"])
        updated += 1
    else:
        skipped += 1

    populated_samples.append(s)

print(f"  Updated : {updated}")
print(f"  Skipped : {skipped} (not yet processed)")

with open(POPULATED_PATH, "w") as f:
    json.dump(populated_samples, f, indent=2)
print(f"Saved → {POPULATED_PATH}")


# ================================================================
# STATS ON POPULATED SAMPLES
# ================================================================

print("\n" + "="*55)
print("  POPULATION STATS")
print("="*55)

done_samples = [s for s in populated_samples
                if s["id"] in checkpoint]

by_action = defaultdict(list)
for s in done_samples:
    by_action[s["action"]].append(s)

for action, slist in by_action.items():
    avg_k = sum(len(s["state"]["known_variables"])
                for s in slist) / max(len(slist), 1)
    avg_m = sum(len(s["state"]["missing_variables"])
                for s in slist) / max(len(slist), 1)
    print(f"\n  {action}")
    print(f"    Avg known_variables  : {avg_k:.2f}")
    print(f"    Avg missing_variables: {avg_m:.2f}")
    print(f"    Samples              : {len(slist)}")

from collections import Counter
all_known   = [v for s in done_samples
               for v in s["state"]["known_variables"]]
all_missing = [v for s in done_samples
               for v in s["state"]["missing_variables"]]

print(f"\n  Top 15 known_variables:")
for v, c in Counter(all_known).most_common(15):
    print(f"    {v:25} : {c}")

print(f"\n  Top 15 missing_variables:")
for v, c in Counter(all_missing).most_common(15):
    print(f"    {v:25} : {c}")


# ================================================================
# INTERACTIVE VIEWER — see what was added
# ================================================================

source_options = ["all"] + \
    list(set(s["metadata"]["source"] for s in done_samples))
action_options = ["all", "ANSWER", "ASK", "ABSTAIN"]

source_dd = widgets.Dropdown(
    options=source_options, value="all",
    description="Source:",
    style={"description_width":"80px"},
    layout=widgets.Layout(width="250px")
)
action_dd = widgets.Dropdown(
    options=action_options, value="ASK",
    description="Action:",
    style={"description_width":"80px"},
    layout=widgets.Layout(width="250px")
)
idx_slider = widgets.IntSlider(
    value=0, min=0, max=100, step=1,
    description="Sample:",
    style={"description_width":"80px"},
    layout=widgets.Layout(width="400px")
)
out = widgets.Output()

ACTION_COLOR = {
    "ANSWER":"#00e676", "ASK":"#40c4ff", "ABSTAIN":"#ff5252"
}

def get_filtered(source, action):
    filtered = done_samples
    if source != "all":
        filtered = [s for s in filtered
                    if s["metadata"]["source"] == source]
    if action != "all":
        filtered = [s for s in filtered
                    if s["action"] == action]
    return filtered

def render_sample(s):
    action  = s["action"]
    color   = ACTION_COLOR.get(action, "#fff")
    known   = s["state"]["known_variables"]
    missing = s["state"]["missing_variables"]
    ctx     = s["context"]["documents"][0]["text"][:400] \
              .replace("\n"," ") if s["context"]["documents"] else ""

    known_html   = "".join(
        f'<span style="background:#1b3a2b; color:#00e676; '
        f'padding:2px 6px; border-radius:4px; margin:2px; '
        f'display:inline-block;">{v}</span>'
        for v in known
    ) or '<span style="color:#666;">— none —</span>'

    missing_html = "".join(
        f'<span style="background:#3a1b1b; color:#ff5252; '
        f'padding:2px 6px; border-radius:4px; margin:2px; '
        f'display:inline-block;">{v}</span>'
        for v in missing
    ) or '<span style="color:#666;">— none —</span>'

    return f"""
    <div style="font-family:monospace; font-size:12px;
                background:#1e1e1e; color:#e0e0e0; padding:14px;
                border-radius:8px; border:1px solid #444;">

      <div style="margin-bottom:6px;">
        <b style="color:#90caf9;">ID:</b> {s['id']}
        &nbsp;|&nbsp;
        <b style="color:#90caf9;">Source:</b> {s['metadata']['source']}
        &nbsp;|&nbsp;
        <b style="color:#90caf9;">Turn:</b> {s['metadata'].get('turn_id','—')}
      </div>

      <div style="background:#263238; padding:8px; border-radius:4px;
                  color:#cfd8dc; margin-bottom:8px; font-size:11px;">
        <b style="color:#90caf9;">CONTEXT:</b> {ctx}...
      </div>

      <div style="margin-bottom:6px;">
        <b style="color:#fff176;">QUERY:</b> {s['query']}
      </div>

      <div style="margin-bottom:8px;">
        <b style="color:#90caf9;">ACTION:</b>
        <span style="color:{color}; font-weight:bold;"> {action}</span>
        &nbsp;&nbsp;
        <b style="color:#90caf9;">failure_mode:</b>
        <span style="color:#ffcc80;">
          {s['state']['failure_mode']}
        </span>
      </div>

      <div style="margin-bottom:6px;">
        <b style="color:#90caf9;">✅ known_variables
          ({len(known)}):</b><br>
        <div style="margin-top:4px;">{known_html}</div>
      </div>

      <div style="margin-bottom:8px;">
        <b style="color:#90caf9;">❓ missing_variables
          ({len(missing)}):</b><br>
        <div style="margin-top:4px;">{missing_html}</div>
      </div>

      <div style="background:#1b3a2b; padding:8px; border-radius:4px;
                  border-left:3px solid {color}; color:#c8e6c9;">
        <b>RESPONSE:</b> {s['response'][:200]}
      </div>
    </div>
    """

def update(change=None):
    filtered = get_filtered(source_dd.value, action_dd.value)
    if not filtered:
        with out:
            clear_output(wait=True)
            print("No samples match filter.")
        return
    idx_slider.max   = len(filtered) - 1
    idx_slider.value = min(idx_slider.value, idx_slider.max)
    s = filtered[idx_slider.value]
    with out:
        clear_output(wait=True)
        print(f"Showing {idx_slider.value+1} of {len(filtered)}")
        display(HTML(render_sample(s)))

source_dd.observe(update, names="value")
action_dd.observe(update, names="value")
idx_slider.observe(update, names="value")

display(widgets.VBox([
    widgets.HTML(
        "<b style='color:#90caf9; font-size:13px;'>"
        "🔍 Variable Population Viewer</b>"
    ),
    widgets.HBox([source_dd, action_dd, idx_slider]),
    out
]))

update()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
from datetime import datetime

# Folder path (update if needed)
folder_path = "/content/unified_dataset"

# Create zip name with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f"unified_dataset_{timestamp}"

# Destination in Drive
drive_path = f"/content/drive/MyDrive/{zip_name}"

# Zip the folder
shutil.make_archive(drive_path, 'zip', folder_path)

print(f"✅ Zipped and saved to: {drive_path}.zip")

In [ ]:
import shutil

zip_path = "/content/drive/MyDrive/unified_dataset_20260331_074625.zip"
local_zip = "/content/unified_dataset.zip"

shutil.copy(zip_path, local_zip)

print("✅ Copied ZIP to Colab")


import zipfile

extract_path = "/content/unified_dataset"

with zipfile.ZipFile(local_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"✅ Extracted to {extract_path}")

In [ ]:
!pip install spacy networkx sentence-transformers matplotlib pyvis tqdm -q
!python -m spacy download en_core_web_sm -q

In [ ]:
import json, re, os, math, pickle
from collections import defaultdict
import spacy
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util
import torch

np.random.seed(42)
spacy.prefer_gpu()
nlp   = spacy.load("en_core_web_sm", disable=["senter"])
sbert = SentenceTransformer("all-MiniLM-L6-v2")

CFG = {
    "sem_threshold"    : 0.50,   # raised from 0.30 — filters noise properly
    "hotpot_max_hops"  : 3,
    "reinforce_answer" : 0.20,
    "reinforce_ask"    : 0.05,
    "penalise_abstain" : 0.10,
    "nlp_batch_size"   : 1024,
    "nlp_n_process"    : 1,
    "max_triples_per_kb": 15,    # cap triples per KB for finetuning dataset
}

In [ ]:
STOPWORDS = {
    "he","she","it","they","his","her","its","their","him","them",
    "this","that","these","those","i","we","you","who","which","what",
    "where","when","how","by","the","a","an","was","were","been",
    "being","have","has","had","do","does","did","will","would",
    "could","should","may","might","shall","can","need","dare","ought",
}

NAMED_ENT_TYPES = {
    "PERSON","ORG","GPE","LOC","WORK_OF_ART",
    "EVENT","LAW","PRODUCT","NORP","FAC","LANGUAGE"
}

SPACY_TO_CAT = {
    "PERSON"     : "Person",
    "ORG"        : "Organization",
    "GPE"        : "Location",
    "LOC"        : "Location",
    "WORK_OF_ART": "Work",
    "EVENT"      : "Event",
    "LAW"        : "Work",
    "PRODUCT"    : "Work",
    "DATE"       : "Attribute",
    "CARDINAL"   : "Attribute",
    "NORP"       : "Concept",
    "FAC"        : "Location",
    "LANGUAGE"   : "Concept",
}

WEAK_VERBS = {
    "be","have","do","get","make","go","come","say","know","think",
    "see","look","want","give","use","find","tell","ask","seem",
    "feel","try","leave","call","keep","let","put","become","show"
}

def normalise(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip().lower()
    text = re.sub(r"[^a-z0-9 \-]", "", text)
    return text

def is_valid_node(text: str) -> bool:
    """Hard filter — rejects pronouns, stopwords, numbers, single chars."""
    if not text or len(text) < 3:
        return False
    if text.lower() in STOPWORDS:
        return False
    if re.fullmatch(r"[\d\s,\.\-]+", text):   # pure numbers
        return False
    return True

def extract_from_doc(doc, source: str):
    """
    FIXED:
    - Only NER entities, NO noun chunks
    - Triples only between two valid named entities
    - No pronouns/stopwords as nodes
    """
    hotpot = (source == "hotpotqa")

    # ── named entities only ───────────────────────────────────────
    seen, ents = set(), []
    ent_token_ids = set()  # track which tokens are part of named entities

    for ent in doc.ents:
        cat  = SPACY_TO_CAT.get(ent.label_, "Concept")
        norm = normalise(ent.text)
        if not is_valid_node(norm):
            continue
        if norm not in seen:
            seen.add(norm)
            ents.append((norm, cat))
        for tok in ent:
            ent_token_ids.add(tok.i)

    # ── triples: ONLY between named entities ──────────────────────
    # build token_idx → normalised entity name map
    tok_to_ent = {}
    for ent in doc.ents:
        norm = normalise(ent.text)
        if is_valid_node(norm):
            for tok in ent:
                tok_to_ent[tok.i] = norm

    triples, seen_triples = [], set()

    for sent in doc.sents:
        for token in sent:
            if token.pos_ != "VERB":
                continue
            lemma = token.lemma_.lower()
            if lemma in WEAK_VERBS and not hotpot:
                continue

            subjs = [c for c in token.children
                     if c.dep_ in {"nsubj","nsubjpass","agent"}]
            objs  = [c for c in token.children
                     if c.dep_ in {"dobj","attr","oprd"}]

            # prep+pobj chains e.g. "born in Rouen"
            for prep in [c for c in token.children if c.dep_ == "prep"]:
                for po in prep.children:
                    if po.dep_ == "pobj":
                        rel = f"{lemma}_{prep.text.lower()}"
                        for s in subjs:
                            sn = tok_to_ent.get(s.i)
                            on = tok_to_ent.get(po.i)
                            # BOTH must be named entities
                            if sn and on and sn != on:
                                key = (sn, rel, on)
                                if key not in seen_triples:
                                    seen_triples.add(key)
                                    triples.append((sn, rel, on, 0.7))

            for s in subjs:
                for o in objs:
                    sn = tok_to_ent.get(s.i)
                    on = tok_to_ent.get(o.i)
                    # BOTH must be named entities
                    if sn and on and sn != on:
                        key = (sn, lemma, on)
                        if key not in seen_triples:
                            seen_triples.add(key)
                            triples.append((sn, lemma, on, 0.8))

    return ents, triples


def triple_to_sentence(subj, rel, obj):
    return f"{subj} {rel.replace('_', ' ')} {obj}"

In [ ]:
class KnowledgeGraph:
    def __init__(self):
        self.G               = nx.DiGraph()
        self.kb_to_nodes     = defaultdict(list)
        self.kb_to_triples   = defaultdict(list)   # NEW: kb_id → clean triples
        self.kb_meta         = {}
        self.node_to_kbs     = defaultdict(set)
        self.variable_nodes  = set()
        self.reinforced_paths= {}

    def add_node(self, node_id, **attrs):
        if not self.G.has_node(node_id):
            self.G.add_node(node_id, **attrs)
        else:
            self.G.nodes[node_id].update(attrs)

    def add_edge(self, u, v, **attrs):
        if self.G.has_edge(u, v):
            ex = self.G[u][v]
            ex["weight"]    = max(ex.get("weight",0), attrs.get("weight",0))
            ex["frequency"] = ex.get("frequency",1) + 1
            ex["sources"]   = list(set(ex.get("sources",[]))|set(attrs.get("sources",[])))
            ex["kb_ids"]    = list(set(ex.get("kb_ids",[])) | set(attrs.get("kb_ids",[])))
        else:
            attrs.setdefault("weight", 1.0)
            attrs.setdefault("frequency", 1)
            self.G.add_edge(u, v, **attrs)

In [ ]:
def build_phase1(kb_entries):
    kg = KnowledgeGraph()

    for entry in kb_entries:
        kb_id  = entry["kb_id"]
        source = entry.get("source","unknown")
        kg.kb_meta[kb_id] = {
            "source"           : source,
            "action_dist"      : entry.get("action_distribution",{}),
            "linked_queries"   : entry.get("linked_queries",[]),
            "linked_actions"   : entry.get("linked_actions",[]),
            "linked_sample_ids": entry.get("linked_sample_ids",[]),
        }

    texts   = [e["text"]                  for e in kb_entries]
    sources = [e.get("source","unknown")  for e in kb_entries]
    kb_ids  = [e["kb_id"]                 for e in kb_entries]

    print(f"Phase 1 — parsing {len(texts)} texts …")
    docs = list(tqdm(
        nlp.pipe(texts, batch_size=CFG["nlp_batch_size"],
                 n_process=CFG["nlp_n_process"]),
        total=len(texts), desc="spaCy"
    ))

    print("Building graph …")
    for doc, source, kb_id in tqdm(zip(docs, sources, kb_ids),
                                    total=len(kb_ids), desc="Graph"):
        ents, triples = extract_from_doc(doc, source)

        # store clean triples per KB for later dataset generation
        kg.kb_to_triples[kb_id] = [
            f"{s} | {r} | {o}" for s,r,o,_ in triples
        ][:CFG["max_triples_per_kb"]]

        for norm, cat in ents:
            kg.add_node(norm, category=cat, label=norm)
            kg.kb_to_nodes[kb_id].append(norm)
            kg.node_to_kbs[norm].add(kb_id)

        for s, r, o, conf in triples:
            for node in (s, o):
                if not kg.G.has_node(node):
                    kg.add_node(node, category="Concept", label=node)
            kg.add_edge(s, o,
                        relation  = r,
                        weight    = conf,
                        sem_score = 0.0,
                        q_score   = 0.0,
                        sources   = [source],
                        kb_ids    = [kb_id])
            kg.kb_to_nodes[kb_id].extend([s, o])

        kg.kb_to_nodes[kb_id] = list(set(kg.kb_to_nodes[kb_id]))

    # prune degree-0 only (we already filtered noise at insert)
    isolated = [n for n in kg.G.nodes() if kg.G.degree(n) == 0]
    kg.G.remove_nodes_from(isolated)

    print(f"  G₀: {kg.G.number_of_nodes()} nodes, "
          f"{kg.G.number_of_edges()} edges")
    return kg

In [ ]:
def phase2_semantic_validation(kg, kb_entries):
    print("Phase 2 — semantic validation …")

    kb_text_map  = {e["kb_id"]: e["text"] for e in kb_entries}
    all_kb_ids   = list(kb_text_map.keys())
    kb_id_to_idx = {kid: i for i, kid in enumerate(all_kb_ids)}

    # ── encode all KB texts — keep on GPU (A100 has room) ─────────
    print(f"  Encoding {len(all_kb_ids)} KB texts …")
    kb_matrix = sbert.encode(
        [kb_text_map[k] for k in all_kb_ids],
        batch_size=2048, show_progress_bar=True,
        convert_to_tensor=True, normalize_embeddings=True
    )  # (105k, 384) — ~162MB on GPU, fine on A100

    # ── build edge list + precompute source KB indices per edge ───
    edge_list   = list(kg.G.edges(data=True))
    sentences   = [triple_to_sentence(u, d.get("relation","related_to"), v)
                   for u,v,d in edge_list]

    # precompute src_idx per edge once — avoids repeated dict lookups
    print("  Precomputing source KB indices per edge …")
    edge_src_idx = []
    for u, v, data in edge_list:
        idx = [kb_id_to_idx[k] for k in data.get("kb_ids",[])
               if k in kb_id_to_idx]
        if not idx:
            idx = [kb_id_to_idx[k]
                   for k in (kg.node_to_kbs.get(u,set()) |
                              kg.node_to_kbs.get(v,set()))
                   if k in kb_id_to_idx]
        edge_src_idx.append(idx)

    # ── chunked encoding + scoring ────────────────────────────────
    CHUNK_SIZE  = 50000   # A100 40GB: safe. 80GB: raise to 50000
    best_scores = np.zeros(len(edge_list), dtype=np.float32)

    print(f"  Encoding + scoring {len(edge_list)} edges "
          f"in chunks of {CHUNK_SIZE} …")

    for start in tqdm(range(0, len(edge_list), CHUNK_SIZE),
                      desc="Edge chunks"):
        end = min(start + CHUNK_SIZE, len(edge_list))

        # encode this chunk of edge sentences on GPU
        chunk_emb = sbert.encode(
            sentences[start:end],
            batch_size=2048, show_progress_bar=False,
            convert_to_tensor=True, normalize_embeddings=True
        )  # (chunk, 384) on GPU

        # score each edge vs its own source KBs
        for i in range(end - start):
            src_idx = edge_src_idx[start + i]
            if not src_idx:
                continue
            # (M, 384) @ (384,) → (M,)  — small, fast
            sims = kb_matrix[src_idx] @ chunk_emb[i]
            best_scores[start + i] = float(sims.max())

        del chunk_emb
        torch.cuda.empty_cache()

    del kb_matrix
    torch.cuda.empty_cache()

    # ── apply scores + filter ──────────────────────────────────────
    print("  Filtering edges …")
    edges_to_remove = []

    for i, (u, v, data) in enumerate(edge_list):
        freq_bonus  = math.log1p(data.get("frequency", 1)) * 0.03
        final_score = float(best_scores[i]) + freq_bonus
        kg.G[u][v]["sem_score"] = round(final_score, 4)
        if final_score < CFG["sem_threshold"]:
            edges_to_remove.append((u, v))

    kg.G.remove_edges_from(edges_to_remove)
    isolated = [n for n in kg.G.nodes()
                if kg.G.degree(n) == 0
                and n not in kg.variable_nodes]
    kg.G.remove_nodes_from(isolated)

    print(f"  G₁: {kg.G.number_of_nodes()} nodes, "
          f"{kg.G.number_of_edges()} edges "
          f"(removed {len(edges_to_remove)} edges, "
          f"{len(isolated)} isolated nodes)")
    return kg

In [ ]:
GENERIC_PHRASES = {
    "the season","the year","the time","the game","the team",
    "the show","the film","one","two","three","the series",
    "the episode","the album","the song","the book"
}

def find_missing_entity(query, kg):
    """
    Only return a missing entity if the query mentions a named
    entity that doesn't exist in the graph.
    Returns the missing entity string or None.
    """
    q_doc  = nlp(query)
    q_ents = [normalise(e.text) for e in q_doc.ents
              if e.label_ in NAMED_ENT_TYPES]
    missing = [e for e in q_ents
               if e and len(e) > 2
               and not kg.G.has_node(e)
               and e not in GENERIC_PHRASES]
    return missing[0] if missing else None

In [ ]:
def phase3_query_refinement(kg, kb_entries):
    print("Phase 3 — query-guided refinement …")
    G_undir = kg.G.to_undirected()

    def fast_path(src, tgt, max_hops):
        try:
            p = nx.shortest_path(G_undir, src, tgt)
            return p if len(p)-1 <= max_hops else None
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            return None

    def inject_variable(anchor, query, sid):
        if not kg.G.has_node(anchor):
            return
        var_id = f"?var_{sid}"
        kg.add_node(var_id, category="Variable", label=var_id,
                    query=query, for_kb=anchor)
        kg.variable_nodes.add(var_id)
        kg.add_edge(anchor, var_id,
                    relation="requires", weight=0.9,
                    sem_score=1.0, q_score=1.0,
                    sources=["variable_injection"], kb_ids=[sid])

    def reinforce_edges(u, v, action):
        if kg.G.has_edge(u, v):
            w = kg.G[u][v]["weight"]
            if action == "ANSWER":
                kg.G[u][v]["weight"]  = min(0.95, w + CFG["reinforce_answer"])
                kg.G[u][v]["q_score"] = kg.G[u][v].get("q_score",0) + 0.3
            elif action == "ASK":
                kg.G[u][v]["weight"]  = min(0.95, w + CFG["reinforce_ask"])
                kg.G[u][v]["q_score"] = kg.G[u][v].get("q_score",0) + 0.1
            elif action == "ABSTAIN":
                kg.G[u][v]["weight"]  = max(0.0, w - CFG["penalise_abstain"])

    # ── 1. collect ALL queries upfront ────────────────────────────
    print("  Collecting all queries …")
    all_queries, all_sids = [], []
    entry_query_map = []   # (entry_idx, query, action, sid)

    for entry_idx, entry in enumerate(kb_entries):
        for q, a, s in zip(entry.get("linked_queries",[]),
                           entry.get("linked_actions",[]),
                           entry.get("linked_sample_ids",[])):
            all_queries.append(q)
            all_sids.append(s)
            entry_query_map.append((entry_idx, q, a, s))

    # ── 2. batch-parse ALL queries at once with nlp.pipe ──────────
    print(f"  Batch-parsing {len(all_queries)} queries …")
    graph_node_set = set(kg.G.nodes())

    sid_to_missing = {}   # sid → missing entity string or None
    for sid, doc in tqdm(
            zip(all_sids,
                nlp.pipe(all_queries,
                         batch_size=CFG["nlp_batch_size"],
                         n_process=CFG["nlp_n_process"])),
            total=len(all_queries), desc="  NLP queries"):
        q_ents = [normalise(e.text) for e in doc.ents
                  if e.label_ in NAMED_ENT_TYPES]
        missing = [e for e in q_ents
                   if e and len(e) > 2
                   and e not in graph_node_set
                   and e not in GENERIC_PHRASES]
        sid_to_missing[sid] = missing[0] if missing else None

    # ── 3. precompute per-KB edge sets ────────────────────────────
    print("  Precomputing per-KB edge sets …")
    kb_edge_map = defaultdict(list)
    for u, v, data in tqdm(kg.G.edges(data=True), desc="  Edge index"):
        for kid in data.get("kb_ids", []):
            kb_edge_map[kid].append((u, v))

    # ── 4. reinforcement loop (no NLP inside) ─────────────────────
    total = len(entry_query_map)
    print(f"  Applying reinforcement on {total} pairs …")

    for entry_idx, query, action, sid in tqdm(entry_query_map,
                                               desc="Reinforcement"):
        entry    = kb_entries[entry_idx]
        kb_id    = entry["kb_id"]
        source   = entry.get("source","unknown")
        max_hops = CFG["hotpot_max_hops"] if source == "hotpotqa" else 2

        kb_nodes = [n for n in kg.kb_to_nodes.get(kb_id,[])
                    if kg.G.has_node(n)]
        kb_edges = kb_edge_map.get(kb_id, [])

        if not kb_nodes:
            continue

        missing_ent = sid_to_missing.get(sid)   # O(1) lookup

        if source == "hotpotqa" and len(kb_nodes) >= 2:
            best_path = None
            for i in range(min(3, len(kb_nodes))):
                for j in range(i+1, min(4, len(kb_nodes))):
                    p = fast_path(kb_nodes[i], kb_nodes[j], max_hops)
                    if p:
                        best_path = p
                        break
                if best_path:
                    break

            if best_path:
                for k in range(len(best_path)-1):
                    reinforce_edges(best_path[k], best_path[k+1], action)
                    reinforce_edges(best_path[k+1], best_path[k], action)
                kg.reinforced_paths[sid] = best_path
                if action == "ASK" and missing_ent:
                    inject_variable(best_path[-1], query, sid)
            else:
                if action == "ASK" and missing_ent:
                    inject_variable(kb_nodes[0], query, sid)
                if action == "ABSTAIN" and kg.G.has_node(kb_nodes[0]):
                    kg.G.nodes[kb_nodes[0]]["missing_info"] = True

        else:
            for u, v in kb_edges:
                reinforce_edges(u, v, action)
            if action == "ASK" and missing_ent:
                inject_variable(kb_nodes[0], query, sid)
            if action == "ABSTAIN" and kb_nodes:
                kg.G.nodes[kb_nodes[0]]["missing_info"] = True

    # ── 5. final weights ──────────────────────────────────────────
    for u, v, data in kg.G.edges(data=True):
        kg.G[u][v]["final_weight"] = round(
            min(1.0, data.get("sem_score",0)*0.5 +
                     data.get("q_score",0)*0.5), 4)

    print(f"  G₂: {kg.G.number_of_nodes()} nodes "
          f"({len(kg.variable_nodes)} var), "
          f"{kg.G.number_of_edges()} edges")
    return kg

In [ ]:
import pickle

def save_checkpoint(kg, phase, path=CKPT_PATH):
    data = {
        "graph"           : kg.G,
        "kb_to_nodes"     : dict(kg.kb_to_nodes),
        "kb_to_triples"   : dict(kg.kb_to_triples),
        "kb_meta"         : kg.kb_meta,
        "node_to_kbs"     : {k: v for k, v in kg.node_to_kbs.items()},
        "variable_nodes"  : kg.variable_nodes,
        "reinforced_paths": kg.reinforced_paths,
    }
    fpath = f"{path}kg_{phase}.pkl"
    with open(fpath, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"  Saved {phase} → {fpath}  ({os.path.getsize(fpath)/1e6:.1f} MB)")


def load_checkpoint(phase, path=CKPT_PATH):
    fpath = f"{path}kg_{phase}.pkl"
    with open(fpath, "rb") as f:
        data = pickle.load(f)
    kg = KnowledgeGraph()
    kg.G                = data["graph"]
    kg.kb_to_nodes      = defaultdict(list, data["kb_to_nodes"])
    kg.kb_to_triples    = defaultdict(list, data.get("kb_to_triples", {}))
    kg.kb_meta          = data["kb_meta"]
    kg.node_to_kbs      = defaultdict(set,  data["node_to_kbs"])
    kg.variable_nodes   = data["variable_nodes"]
    kg.reinforced_paths = data["reinforced_paths"]
    print(f"  Loaded {phase} ← {fpath}  "
          f"({kg.G.number_of_nodes()} nodes, {kg.G.number_of_edges()} edges)")
    return kg

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CKPT_PATH = "/content/drive/MyDrive/kg_project/"
os.makedirs(CKPT_PATH, exist_ok=True)

KB_PATH = "/content/rag_pipeline/knowledge_base.json"
with open(KB_PATH) as f:
    kb_entries = json.load(f)
print(f"Loaded {len(kb_entries)} KB entries")

def graph_stats(kg, label="G"):
    G = kg.G
    print(f"\n{'='*50}  {label}")
    print(f"  Nodes : {G.number_of_nodes()}")
    print(f"  Edges : {G.number_of_edges()}")
    print(f"  Vars  : {len(kg.variable_nodes)}")
    if G.number_of_edges() > 0:
        w = [d.get("final_weight", d.get("weight",0)) for _,_,d in G.edges(data=True)]
        print(f"  Avg weight : {np.mean(w):.3f}")
    cats = defaultdict(int)
    for _, d in G.nodes(data=True):
        cats[d.get("category","?")] += 1
    for cat, cnt in sorted(cats.items(), key=lambda x:-x[1]):
        print(f"    {cat:20s}: {cnt}")

# ── Phase 1
if os.path.exists(f"{CKPT_PATH}kg_G0.pkl"):
    kg = load_checkpoint("G0", CKPT_PATH)
else:
    kg = build_phase1(kb_entries)
    save_checkpoint(kg, "G0", CKPT_PATH)
graph_stats(kg, "G₀ — raw")



In [ ]:
# ── Phase 2
if os.path.exists(f"{CKPT_PATH}kg_G1.pkl"):
    kg = load_checkpoint("G1", CKPT_PATH)
else:
    kg = phase2_semantic_validation(kg, kb_entries)
    save_checkpoint(kg, "G1", CKPT_PATH)
graph_stats(kg, "G₁ — semantic")


In [ ]:
# delete only G2, keep G0 and G1
import os
g2_path = f"{CKPT_PATH}kg_G2.pkl"
if os.path.exists(g2_path):
    os.remove(g2_path)
    print("G2 deleted — G0 and G1 preserved")

# then rerun just this block
kg = load_checkpoint("G1", CKPT_PATH)   # loads in seconds
kg = phase3_query_refinement(kg, kb_entries)
save_checkpoint(kg, "G2", CKPT_PATH)
graph_stats(kg, "G₂ — decision-aware")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter, defaultdict
import networkx as nx
import re
from difflib import SequenceMatcher

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 30)

CAT_COLORS = {
    "Person"       : "#4E9AF1",
    "Organization" : "#F4A261",
    "Location"     : "#57CC99",
    "Work"         : "#E76F51",
    "Event"        : "#9B72CF",
    "Concept"      : "#A8DADC",
    "Attribute"    : "#FFD166",
    "Variable"     : "#FF6B6B",
}

# ═══════════════════════════════════════════════════════════════
def kg_eda_full(kg, kb_entries):

    G           = kg.G
    kb_text_map = {e["kb_id"]: e["text"] for e in kb_entries}

    # separate real nodes from variable placeholders
    real_nodes = [n for n in G.nodes() if not str(n).startswith("?var_")]
    var_nodes  = [n for n in G.nodes() if str(n).startswith("?var_")]
    real_G     = G.subgraph(real_nodes).copy()

    print("="*65)
    print("  KG EDA REPORT")
    print("="*65)

    # ── 1. basic stats ────────────────────────────────────────────
    print("\n📊  BASIC STATS")
    degrees     = [d for _,d in real_G.degree()]
    weights_real= [d.get("final_weight", d.get("weight",0))
                   for _,_,d in real_G.edges(data=True)]
    stats = [
        ("Total nodes (incl. ?var)",     G.number_of_nodes()),
        ("Real entity nodes",            len(real_nodes)),
        ("Variable (?var) nodes",        len(var_nodes)),
        ("Var node %",                   f"{len(var_nodes)/max(G.number_of_nodes(),1)*100:.1f}%"
                                         + ("  ⚠️ HIGH" if len(var_nodes)/G.number_of_nodes()>0.5 else "  ✅")),
        ("Total edges",                  G.number_of_edges()),
        ("Real→Real edges",              real_G.number_of_edges()),
        ("Avg degree (real nodes)",      f"{np.mean(degrees):.2f}" if degrees else "0"),
        ("Avg weight (real edges only)", f"{np.mean(weights_real):.3f}" if weights_real else "0"),
        ("Weakly connected components",  nx.number_weakly_connected_components(real_G)),
        ("Largest WCC",                  max((len(c) for c in nx.weakly_connected_components(real_G)),
                                         default=0)),
    ]
    print(pd.DataFrame(stats, columns=["Metric","Value"]).to_string(index=False))

    # ── 2. category breakdown ─────────────────────────────────────
    print("\n\n📦  NODE CATEGORIES (real nodes only)")
    cats = Counter(G.nodes[n].get("category","?") for n in real_nodes)
    cat_df = pd.DataFrame(cats.most_common(), columns=["Category","Count"])
    cat_df["%"] = (cat_df["Count"]/max(len(real_nodes),1)*100).round(1)
    print(cat_df.to_string(index=False))

    # ── 3. relation types ─────────────────────────────────────────
    print("\n\n🔗  TOP 20 RELATION TYPES (real edges)")
    weak = {"is","has","was","be","have","related_to",
            "implicit_link","query_linked","requires"}
    rels = Counter(d.get("relation","?")
                   for _,_,d in real_G.edges(data=True))
    total_e = real_G.number_of_edges()
    weak_ct = sum(v for k,v in rels.items() if k in weak)
    print(f"  Weak/generic relation % : "
          f"{weak_ct/max(total_e,1)*100:.1f}%  "
          f"{'⚠️ HIGH' if weak_ct/max(total_e,1)>0.3 else '✅ OK'}")
    rel_df = pd.DataFrame(rels.most_common(20), columns=["Relation","Count"])
    rel_df["%"] = (rel_df["Count"]/max(total_e,1)*100).round(1)
    rel_df["flag"] = rel_df["Relation"].apply(
        lambda r: "⚠️" if r in weak else "✅")
    print(rel_df.to_string(index=False))

    # ── 4. weight distribution (real edges) ───────────────────────
    print("\n\n⚖️   WEIGHT DISTRIBUTION (real edges only)")
    if weights_real:
        w = pd.Series(weights_real)
        bins   = [0,.2,.4,.6,.8,1.01]
        labels = ["0.0–0.2","0.2–0.4","0.4–0.6","0.6–0.8","0.8–1.0"]
        w_df   = pd.cut(w, bins=bins, labels=labels).value_counts().sort_index()
        w_df   = w_df.reset_index()
        w_df.columns = ["Bucket","Count"]
        w_df["%"] = (w_df["Count"]/len(weights_real)*100).round(1)
        print(w_df.to_string(index=False))
        print(f"\n  mean={w.mean():.3f}  median={w.median():.3f}  "
              f"std={w.std():.3f}  min={w.min():.3f}  max={w.max():.3f}")

    # ── 5. top hub nodes ──────────────────────────────────────────
    print("\n\n🏆  TOP 20 HUB NODES (real graph)")
    bad = {"he","she","it","they","by","which","this","that","the","a","an"}
    rows = []
    for n, deg in sorted(real_G.degree(), key=lambda x:-x[1])[:20]:
        cat  = G.nodes[n].get("category","?")
        flag = "⚠️ NOISE" if n.lower() in bad else \
               "⚠️ SHORT" if len(n)<=2 else \
               "⚠️ NUM"   if re.fullmatch(r"[\d\s,\.]+",n) else "✅"
        rows.append((n, cat, deg,
                     real_G.in_degree(n), real_G.out_degree(n), flag))
    print(pd.DataFrame(rows,
          columns=["Node","Category","Degree","In","Out","Flag"])
          .to_string(index=False))

    # ── 6. sample triples ─────────────────────────────────────────
    print("\n\n🔍  SAMPLE HIGH-CONFIDENCE TRIPLES (weight ≥ 0.6)")
    triples = [(u, d.get("relation","?"), v,
                round(d.get("final_weight", d.get("weight",0)),3),
                d.get("sources",["?"])[0])
               for u,v,d in real_G.edges(data=True)
               if d.get("final_weight", d.get("weight",0)) >= 0.6][:20]
    if triples:
        print(pd.DataFrame(triples,
              columns=["Subject","Relation","Object","Weight","Source"])
              .to_string(index=False))
    else:
        print("  ⚠️  No edges with weight ≥ 0.6 — threshold may be too aggressive")

    # ── 7. source breakdown ───────────────────────────────────────
    print("\n\n📁  SOURCE BREAKDOWN")
    src_ct = Counter(m.get("source","?") for m in kg.kb_meta.values())
    act_by_src = defaultdict(Counter)
    for meta in kg.kb_meta.values():
        src = meta.get("source","?")
        for a in meta.get("linked_actions",[]):
            act_by_src[src][a] += 1
    rows = []
    for src, cnt in src_ct.most_common():
        ac = act_by_src[src]
        rows.append((src, cnt,
                     ac.get("ANSWER",0), ac.get("ASK",0), ac.get("ABSTAIN",0)))
    print(pd.DataFrame(rows,
          columns=["Source","KB entries","ANSWER","ASK","ABSTAIN"])
          .to_string(index=False))

    # ── 8. entity duplicate check ─────────────────────────────────
    print("\n\n🔁  NEAR-DUPLICATE ENTITY CHECK (top 100 nodes)")
    top100 = [n for n,_ in sorted(real_G.degree(),
              key=lambda x:-x[1])[:100]]
    dup_clusters, visited = [], set()
    for i, n1 in enumerate(top100):
        if n1 in visited: continue
        cl = [n1]
        for n2 in top100[i+1:]:
            if n2 in visited: continue
            if SequenceMatcher(None,n1,n2).ratio() >= 0.82 and n1!=n2:
                cl.append(n2); visited.add(n2)
        if len(cl)>1: dup_clusters.append(cl)
        visited.add(n1)
    print(f"  Duplicate clusters : {len(dup_clusters)}  "
          f"{'⚠️ HIGH' if len(dup_clusters)>10 else '✅ OK'}")
    for cl in dup_clusters[:8]:
        print(f"    {' | '.join(cl)}")

    # ── 9. var node sample ────────────────────────────────────────
    print(f"\n\n❓  VARIABLE NODE SAMPLE (ASK placeholders) — "
          f"total={len(var_nodes)}")
    rows = []
    for vn in var_nodes[:12]:
        d = G.nodes[vn]
        rows.append({
            "var_id" : vn[:35],
            "anchor" : d.get("for_kb","?")[:25],
            "query"  : d.get("query","?")[:55],
        })
    print(pd.DataFrame(rows).to_string(index=False))

    # ── 10. local subgraph inspection ─────────────────────────────
    print("\n\n🔬  LOCAL SUBGRAPH INSPECTION (top 5 hubs)")
    for node, deg in sorted(real_G.degree(), key=lambda x:-x[1])[:5]:
        out_e = [(v, real_G[node][v].get("relation","?"),
                  round(real_G[node][v].get("final_weight",0),3))
                 for v in real_G.successors(node)][:5]
        in_e  = [(u, real_G[u][node].get("relation","?"),
                  round(real_G[u][node].get("final_weight",0),3))
                 for u in real_G.predecessors(node)][:3]
        src_text = ""
        for kb_id in list(kg.node_to_kbs.get(node,set()))[:1]:
            src_text = kb_text_map.get(kb_id,"")[:100]
        print(f"\n  Node   : {node}  [{G.nodes[node].get('category','?')}]"
              f"  degree={deg}")
        print(f"  → OUT  : {out_e}")
        print(f"  ← IN   : {in_e}")
        print(f"  Source : {src_text}…")

    print("\n" + "="*65)


# ═══════════════════════════════════════════════════════════════
def kg_plots(kg):
    """6-panel summary plot — real nodes/edges only."""
    G          = kg.G
    real_nodes = [n for n in G.nodes() if not str(n).startswith("?var_")]
    real_G     = G.subgraph(real_nodes).copy()

    fig, axes = plt.subplots(2, 3, figsize=(18,10))
    fig.patch.set_facecolor("#1a1a2e")
    for ax in axes.flat:
        ax.set_facecolor("#1a1a2e")
        for spine in ax.spines.values():
            spine.set_edgecolor("#444")
        ax.tick_params(colors="white")

    def wlabel(ax):
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_color("white")

    # 1. category bar
    ax = axes[0,0]
    cats = Counter(G.nodes[n].get("category","?") for n in real_nodes)
    cats.pop("Variable", None)
    names, vals = zip(*cats.most_common()) if cats else ([],[])
    colors = [CAT_COLORS.get(n,"#AAAAAA") for n in names]
    ax.barh(names, vals, color=colors)
    ax.set_title("Node Categories (real)", color="white")
    wlabel(ax)

    # 2. top 15 relations
    ax = axes[0,1]
    rels = Counter(d.get("relation","?") for _,_,d in real_G.edges(data=True))
    top_rels = rels.most_common(15)
    ax.barh([r for r,_ in top_rels], [c for _,c in top_rels],
            color="#4E9AF1")
    ax.set_title("Top 15 Relations (real edges)", color="white")
    wlabel(ax)

    # 3. weight histogram
    ax = axes[0,2]
    weights = [d.get("final_weight", d.get("weight",0))
               for _,_,d in real_G.edges(data=True)]
    if weights:
        ax.hist(weights, bins=25, color="#57CC99", edgecolor="#1a1a2e")
    ax.set_title("Edge Weight Distribution", color="white")
    ax.set_xlabel("final_weight", color="white")
    wlabel(ax)

    # 4. action distribution pie
    ax = axes[1,0]
    act = Counter()
    for meta in kg.kb_meta.values():
        for a in meta.get("linked_actions",[]):
            act[a] += 1
    if act:
        ax.pie(act.values(), labels=act.keys(),
               colors=["#57CC99","#4E9AF1","#FF6B6B"],
               autopct="%1.1f%%",
               textprops={"color":"white"})
    ax.set_title("Action Distribution", color="white")

    # 5. source breakdown
    ax = axes[1,1]
    src = Counter(m.get("source","?") for m in kg.kb_meta.values())
    ax.bar(src.keys(), src.values(),
           color=["#F4A261","#9B72CF","#57CC99","#4E9AF1"])
    ax.set_title("KB Entries per Source", color="white")
    wlabel(ax)

    # 6. degree distribution log scale
    ax = axes[1,2]
    degrees = [d for _,d in real_G.degree() if d > 0]
    if degrees:
        ax.hist(degrees, bins=40, color="#FFD166",
                edgecolor="#1a1a2e", log=True)
    ax.set_title("Degree Distribution (log)", color="white")
    ax.set_xlabel("degree", color="white")
    ax.set_ylabel("count", color="white")
    wlabel(ax)

    plt.suptitle("Knowledge Graph — EDA Overview (real nodes only)",
                 color="white", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(f"{CKPT_PATH}kg_eda_plots.png",
                dpi=130, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.show()
    print("Saved → kg_eda_plots.png")


# ═══════════════════════════════════════════════════════════════
def sample_graph_plot(kg, n_nodes=30, seed_node=None):
    """
    Plot a small clean subgraph of n_nodes around a seed.
    If seed_node=None, picks the highest-degree real node.
    """
    G          = kg.G
    real_nodes = [n for n in G.nodes() if not str(n).startswith("?var_")]
    real_G     = G.subgraph(real_nodes).copy()

    if not real_G.nodes():
        print("No real nodes to plot.")
        return

    # pick seed
    if seed_node is None or seed_node not in real_G:
        seed_node = max(real_G.degree(), key=lambda x: x[1])[0]

    # BFS ego graph
    ego = nx.ego_graph(real_G.to_undirected(),
                       seed_node, radius=2)
    nodes_to_show = list(ego.nodes())[:n_nodes]
    subG = real_G.subgraph(nodes_to_show).copy()

    if subG.number_of_nodes() < 2:
        print(f"Too few nodes around '{seed_node}'. Try a higher-degree node.")
        return

    pos = nx.spring_layout(subG, seed=42, k=2.0, iterations=80)

    fig, ax = plt.subplots(figsize=(14, 10))
    ax.set_facecolor("#1a1a2e")
    fig.patch.set_facecolor("#1a1a2e")

    # edges
    for u, v, data in subG.edges(data=True):
        fw    = data.get("final_weight", data.get("weight", 0.5))
        rel   = data.get("relation","")
        color = ("#FF6B6B" if rel == "requires" else
                 "#555555" if rel == "implicit_link" else
                 "#4fc978")
        nx.draw_networkx_edges(
            subG, pos, edgelist=[(u,v)],
            edge_color=[color], width=max(0.5, fw*3),
            alpha=0.7, arrows=True, arrowsize=15, ax=ax,
            connectionstyle="arc3,rad=0.08"
        )

    # edge labels (relation name) on stronger edges only
    edge_labels = {(u,v): data.get("relation","")[:12]
                   for u,v,data in subG.edges(data=True)
                   if data.get("final_weight", data.get("weight",0)) >= 0.5
                   and data.get("relation") not in
                       ("implicit_link","query_linked","requires")}
    nx.draw_networkx_edge_labels(
        subG, pos, edge_labels=edge_labels,
        font_size=6, font_color="#cccccc", ax=ax
    )

    # nodes
    for cat, color in CAT_COLORS.items():
        nodelist = [n for n in subG.nodes()
                    if subG.nodes[n].get("category") == cat]
        if not nodelist: continue
        sizes = [max(300, subG.degree(n) * 150) for n in nodelist]
        nx.draw_networkx_nodes(
            subG, pos, nodelist=nodelist,
            node_color=color, node_size=sizes,
            alpha=0.95, ax=ax
        )

    # node labels
    nx.draw_networkx_labels(
        subG, pos,
        labels={n: n[:18] for n in subG.nodes()},
        font_size=7, font_color="white", ax=ax
    )

    # legend
    legend = [mpatches.Patch(color=c, label=cat)
              for cat, c in CAT_COLORS.items()
              if any(subG.nodes[n].get("category")==cat
                     for n in subG.nodes())]
    ax.legend(handles=legend, loc="upper left",
              fontsize=8, framealpha=0.3,
              facecolor="#ffffff22", labelcolor="white")

    ax.set_title(f"Sample Subgraph — seed: '{seed_node}'  "
                 f"({subG.number_of_nodes()} nodes, "
                 f"{subG.number_of_edges()} edges)",
                 color="white", fontsize=12)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(f"{CKPT_PATH}kg_sample_graph.png",
                dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.show()
    print(f"Saved → kg_sample_graph.png  (seed='{seed_node}')")

In [ ]:
# ── full EDA report
kg_eda_full(kg, kb_entries)

# ── 6-panel plots
kg_plots(kg)

# ── sample graph — auto picks highest degree node
sample_graph_plot(kg, n_nodes=30)

In [ ]:
def postprocess_kg(kg, kb_entries):
    """
    Cleans G2 in-place without rerunning any phase.
    All thresholds are data-driven, not hardcoded.
    """
    G = kg.G
    print("="*60)
    print("  KG POST-PROCESSING")
    print("="*60)

    kb_text_map = {e["kb_id"]: e["text"] for e in kb_entries}

    # ═══════════════════════════════════════════════════════════
    # STEP 1 — Detect & remove noise nodes dynamically
    # Strategy: a node is noise if it looks like a stopword,
    # pure number, or is suspiciously short AND low-information
    # (not a real named entity based on spaCy)
    # ═══════════════════════════════════════════════════════════
    print("\n  STEP 1 — Noise node detection …")

    real_nodes = [n for n in G.nodes() if not str(n).startswith("?var_")]

    # batch-NER the node names themselves to check if spaCy
    # considers them real named entities
    print(f"    Checking {len(real_nodes)} nodes via spaCy NER …")
    node_is_named = {}
    for doc, node in zip(
            nlp.pipe(real_nodes, batch_size=CFG["nlp_batch_size"]),
            real_nodes):
        # a node is "named" if spaCy finds at least one named entity
        # covering most of the text
        ents = [e for e in doc.ents if e.label_ in NAMED_ENT_TYPES]
        node_is_named[node] = len(ents) > 0

    # also flag nodes that are:
    # - pure numbers / dates with no alpha
    # - single common words (degree > threshold but no named entity)
    # - suspiciously generic (detected dynamically via POS)
    noise_nodes = []
    degrees     = dict(G.degree())
    avg_degree  = np.mean(list(degrees.values()))

    for node in real_nodes:
        norm = node.lower().strip()
        # rule 1: pure numeric
        if re.fullmatch(r"[\d\s,\.\-\/]+", norm):
            noise_nodes.append((node, "pure_numeric"))
            continue
        # rule 2: single token AND not a named entity AND short
        if " " not in norm and len(norm) <= 4 and not node_is_named.get(node):
            noise_nodes.append((node, "short_non_entity"))
            continue
        # rule 3: high degree hub but spaCy says NOT a named entity
        # → likely a generic phrase that slipped through
        if degrees.get(node, 0) > avg_degree * 3 and not node_is_named.get(node):
            noise_nodes.append((node, "generic_hub"))
            continue
        # rule 4: starts with "the " AND not named entity
        if norm.startswith("the ") and not node_is_named.get(node):
            noise_nodes.append((node, "generic_the_phrase"))
            continue

    print(f"    Noise nodes detected: {len(noise_nodes)}")
    noise_df = pd.DataFrame(noise_nodes[:30], columns=["Node","Reason"])
    print(noise_df.to_string(index=False))

    # remove noise nodes
    noise_set = {n for n,_ in noise_nodes}
    G.remove_nodes_from(noise_set)
    print(f"    Removed {len(noise_set)} noise nodes")

    # ═══════════════════════════════════════════════════════════
    # STEP 2 — Fix bad variable node anchors
    # A variable anchor is "bad" if:
    # - the anchor node was removed (noise)
    # - the anchor is not a named entity
    # - the anchor has no semantic relation to the query
    # Fix: re-anchor to best matching node from the same KB
    # ═══════════════════════════════════════════════════════════
    print("\n  STEP 2 — Fixing bad variable node anchors …")

    bad_var_anchors = 0
    removed_vars    = 0
    reanchored      = 0

    # encode all current real nodes once for reanchoring
    current_real = [n for n in G.nodes()
                    if not str(n).startswith("?var_")]
    if current_real:
        node_embs = sbert.encode(
            current_real, batch_size=1024,
            convert_to_tensor=True, normalize_embeddings=True,
            show_progress_bar=False
        )
        node_list = current_real
    else:
        node_embs = None
        node_list = []

    var_nodes_copy = list(kg.variable_nodes)

    for var_id in tqdm(var_nodes_copy, desc="    Var anchors"):
        if not G.has_node(var_id):
            kg.variable_nodes.discard(var_id)
            continue

        vdata  = G.nodes[var_id]
        anchor = vdata.get("for_kb", "")
        query  = vdata.get("query", "")

        # check if anchor still exists and is valid
        anchor_ok = (G.has_node(anchor)
                     and not str(anchor).startswith("?var_")
                     and node_is_named.get(anchor, True))

        if not anchor_ok:
            bad_var_anchors += 1

            if node_embs is not None and query:
                # find best real node for this query
                q_emb = sbert.encode(
                    query, convert_to_tensor=True,
                    normalize_embeddings=True,
                    show_progress_bar=False
                )
                sims    = (node_embs @ q_emb).cpu().numpy()
                best_i  = int(sims.argmax())
                best_sim= float(sims[best_i])
                new_anchor = node_list[best_i]

                if best_sim >= 0.3 and G.has_node(new_anchor):
                    # reanchor: remove old edge, add new one
                    old_preds = list(G.predecessors(var_id))
                    for pred in old_preds:
                        if G.has_edge(pred, var_id):
                            G.remove_edge(pred, var_id)

                    G.nodes[var_id]["for_kb"] = new_anchor
                    kg.add_edge(new_anchor, var_id,
                                relation  = "requires",
                                weight    = 0.8,
                                sem_score = 0.8,
                                q_score   = 0.8,
                                sources   = ["reanchored"],
                                kb_ids    = [var_id])
                    reanchored += 1
                else:
                    # similarity too low — this var node is meaningless
                    G.remove_node(var_id)
                    kg.variable_nodes.discard(var_id)
                    removed_vars += 1
            else:
                G.remove_node(var_id)
                kg.variable_nodes.discard(var_id)
                removed_vars += 1

    print(f"    Bad anchors found  : {bad_var_anchors}")
    print(f"    Re-anchored        : {reanchored}")
    print(f"    Removed (no match) : {removed_vars}")

    # ═══════════════════════════════════════════════════════════
    # STEP 3 — Remove isolated nodes (degree 0 after cleanup)
    # ═══════════════════════════════════════════════════════════
    print("\n  STEP 3 — Removing isolated nodes …")
    isolated = [n for n in G.nodes()
                if G.degree(n) == 0
                and not str(n).startswith("?var_")]
    G.remove_nodes_from(isolated)
    print(f"    Removed {len(isolated)} isolated nodes")

    # ═══════════════════════════════════════════════════════════
    # STEP 4 — Recompute final weights (clean)
    # ═══════════════════════════════════════════════════════════
    print("\n  STEP 4 — Recomputing final weights …")
    for u, v, data in G.edges(data=True):
        sem = min(data.get("sem_score", 0), 0.95)
        q   = min(data.get("q_score",   0), 0.95)
        G[u][v]["final_weight"] = round(min(0.95, sem*0.5 + q*0.5), 4)

    # ═══════════════════════════════════════════════════════════
    # STEP 5 — Reindex kb_to_nodes (remove pruned nodes)
    # ═══════════════════════════════════════════════════════════
    print("\n  STEP 5 — Reindexing kb_to_nodes …")
    for kb_id in kg.kb_to_nodes:
        kg.kb_to_nodes[kb_id] = [n for n in kg.kb_to_nodes[kb_id]
                                  if G.has_node(n)]
    for node in list(kg.node_to_kbs.keys()):
        if not G.has_node(node):
            del kg.node_to_kbs[node]

    print(f"\n  ✅ Post-processing complete")
    print(f"     Nodes : {G.number_of_nodes()}  "
          f"(real={sum(1 for n in G.nodes() if not str(n).startswith('?var_'))}, "
          f"var={len(kg.variable_nodes)})")
    print(f"     Edges : {G.number_of_edges()}")
    return kg

In [ ]:
# load G2 if not already in memory
kg = load_checkpoint("G2", CKPT_PATH)

# post-process
kg = postprocess_kg(kg, kb_entries)

# save cleaned version
save_checkpoint(kg, "G2_clean", CKPT_PATH)

# re-run EDA on cleaned graph
kg_eda_full(kg, kb_entries)
kg_plots(kg)
sample_graph_plot(kg, n_nodes=30)